# Suite SKL — Agent guides

`AGENTS.md`, `llms.txt` and the skill teach AI agents what this package is for and how to use it. The skill is written
once, in `skills/mbse-expressions/`, and `skills/sync.sh` copies it into this package, in `skill/`. It holds no
programs: each how it teaches links its case in each language's tutorials or tests, which run as tests themselves.
Stale guidance misleads more than missing guidance, so these cases check that the packaged copy of the skill is
current, and that every link in the guides resolves, each case it links in both languages but those
`docs/EQUIVALENCE.md` lists as one language's only. SKL-02, which ran the skill's programs, is retired.

A how the skill teaches without a case is untested: a gap, which is a bug. So every section of the skill (a
`##` or `###` heading and its text, up to the next heading of its level or higher) links a case in each language,
or says why it teaches no how in a comment on the line after its heading, `<!-- no case: reason -->`; a section
whose subsections each pass passes.

The tutorials are cases too, each step anchored as the skill links it, and SKL-05 checks that both languages have
the same tutorials, with the same titles and the same steps in the same order.

In [ ]:
import { existsSync, readdirSync, readFileSync, statSync } from "node:fs";
import { join, relative, resolve } from "node:path";

import { assert } from "./support.js";

const ROOT = resolve("../..");
const SKILL = join(ROOT, "skills", "mbse-expressions");
const PACKAGED = resolve("../skill");

function walk(folder: string): string[] {
  return readdirSync(folder).flatMap((name) => {
    const path = join(folder, name);
    return statSync(path).isDirectory() ? walk(path) : [path];
  });
}

/** The files under `folder`, by relative path, in code-point order. */
function files(folder: string): [string, string][] {
  return walk(folder).map((path): [string, string] => [relative(folder, path).split("\\").join("/"), readFileSync(path, "latin1")])
    .sort(([a], [b]) => (a < b ? -1 : a > b ? 1 : 0));
}

const GUIDES = [join(ROOT, "AGENTS.md"), join(ROOT, "llms.txt"),
  ...files(SKILL).filter(([name]) => name.endsWith(".md")).map(([name]) => join(SKILL, name))];

const LINKS = 652;

<a id="SKL-01"></a>
## SKL-01 · The packaged copy of the skill is current

In [ ]:
assert(files(SKILL).some(([name]) => name === "SKILL.md"));
assert(JSON.stringify(files(PACKAGED)) === JSON.stringify(files(SKILL)), "the packaged skill is stale: run skills/sync.sh");

<a id="SKL-03"></a>
## SKL-03 · Every link in the agent guides resolves, anchors included

In [ ]:
{
  /** GitHub's anchor for a heading. */
  const slug = (heading: string) => heading.trim().toLowerCase().replace(/[^\w\- ]/g, "").split(" ").join("-");
  /** A document's markdown: a notebook's markdown cells, or the file. */
  const markdown = (path: string) => !path.endsWith(".ipynb") ? readFileSync(path, "utf8")
    : (JSON.parse(readFileSync(path, "utf8")).cells as { cell_type: string; source: string | string[] }[])
      .filter((cell) => cell.cell_type === "markdown")
      .map((cell) => (Array.isArray(cell.source) ? cell.source.join("") : cell.source)).join("\n");
  /** A document's `<a id>`s: an item's, or a notebook case's. */
  const ids = (path: string) => new Set([...markdown(path).matchAll(/<a id="([^"]+)"><\/a>/g)].map((m) => m[1] as string));
  /** A document's anchors: its headings' and its `<a id>`s. */
  const anchors = (path: string) => new Set([...[...markdown(path).matchAll(/^#+ (.*)$/gm)].map((m) => slug(m[1] as string)),
    ...ids(path)]);
  // The cases EQUIVALENCE.md lists as one language's only, which have no twin in the other: none yet.
  const equivalence = readFileSync(join(ROOT, "docs", "EQUIVALENCE.md"), "utf8");
  const section = equivalence.split("\n## Cases in one language only\n");
  assert(section.length === 2, "EQUIVALENCE.md has no section of cases in one language only");
  const oneLanguage = (section[1] as string).split("\n## ")[0] as string;
  const ONE_LANGUAGE = new Set([...oneLanguage.matchAll(/`([A-Z]+-\d+[a-z]?|T\d\d\.\d+)`/g)].map((m) => m[1] as string));
  const TWINS: Record<string, string> = { "python3/": "typescript5/", "typescript5/": "python3/" };
  const problems: string[] = [];
  let [checked, cases] = [0, 0];
  for (const guide of GUIDES) {
    for (const [, target] of readFileSync(guide, "utf8").matchAll(/\]\(([^)\s]+)\)/g) as IterableIterator<[string, string]>) {
      let path: string, base: string;
      const sibling = /^(?:mbse:([^/]+)|https:\/\/github\.com\/pitaman71\/([^/]+)\/blob\/main)\/(.*)$/.exec(target);
      const repo = sibling ? (sibling[1] ?? sibling[2]) as string : ""; // the sibling checkout, or this one, whatever its folder's name
      if (sibling) [path, base] = [sibling[3] as string, repo === "mbse-expressions" ? ROOT : join(ROOT, "..", repo)];
      else if (target.includes("://")) continue;
      else [path, base] = [target, resolve(guide, "..")];
      const [name, anchor] = path.split("#") as [string, string | undefined];
      const resolved = name ? join(base, name) : guide; // a bare anchor is the guide's own
      checked += 1;
      const label = `${relative(resolve(guide, ".."), guide)}: ${target}`;
      if (!existsSync(resolved)) problems.push(label);
      else if (anchor && !anchors(resolved).has(anchor)) problems.push(label);
      else if (anchor && resolved.endsWith(".ipynb")) { // a notebook case: its <a id>, here and in its twin
        cases += 1;
        const lang = Object.keys(TWINS).find((prefix) => name.startsWith(prefix));
        const twin = lang === undefined ? null : join(base, (TWINS[lang] as string) + name.slice(lang.length));
        if (!ids(resolved).has(anchor)) problems.push(`${label}: no <a id> ${anchor}`);
        if ((twin === null || !existsSync(twin) || !ids(twin).has(anchor)) && !ONE_LANGUAGE.has(anchor)) {
          problems.push(`${label}: no ${anchor} in the other language's twin`);
        }
      }
    }
  }
  assert(problems.length === 0 && cases > 0, problems.join("; "));
  assert(checked === LINKS, String(checked));
}

<a id="SKL-04"></a>
## SKL-04 · Every section of the skill links a case in each language, or says why it needs none

In [ ]:
{
  const CASE = /\]\(mbse:[^/]+\/(python3|typescript5)\/[^)\s#]+\.ipynb#[^)\s]+\)/g; // a case, in one language
  const OPT_OUT = /^<!-- no case: \S.* -->$/;
  /** A guide's sections: each `##` or `###` heading's line, its title, its text up to the next heading of its level
   * or higher, and its subsections' lines. */
  const sections = (text: string) => {
    const lines = text.split("\n");
    const headings: [number, number, string][] = [];
    let fenced = false;
    for (const [number, line] of lines.entries()) {
      const heading = /^(#+) (.*)$/.exec(line);
      if (line.startsWith("```")) fenced = !fenced;
      else if (!fenced && heading) headings.push([number, (heading[1] as string).length, heading[2] as string]);
    }
    return headings.flatMap(([number, level, title], k) => {
      if (level !== 2 && level !== 3) return [];
      const later = headings.slice(k + 1);
      const end = later.find(([, deeper]) => deeper <= level)?.[0] ?? lines.length;
      const subsections = level === 2 ? later.filter(([n, deeper]) => n < end && deeper === 3).map(([n]) => n) : [];
      return [{ number, title, text: lines.slice(number + 1, end), subsections }];
    });
  };
  // A link counts for the language whose notebook it links, so a case EQUIVALENCE.md lists as one language's only
  // counts for that language alone.
  const gaps: string[] = [];
  let counted = 0;
  for (const guide of GUIDES.filter((path) => !relative(SKILL, path).startsWith(".."))) {
    const passes = new Map<number, boolean>();
    for (const { number, title, text, subsections } of sections(readFileSync(guide, "utf8")).reverse()) {
      const languages = new Set([...text.join("\n").matchAll(CASE)].map((link) => link[1] as string));
      const optedOut = text.length > 0 && OPT_OUT.test(text[0] as string);
      passes.set(number, optedOut || (languages.has("python3") && languages.has("typescript5"))
        || (subsections.length > 0 && subsections.every((n) => passes.get(n) === true)));
      counted += 1;
      if (passes.get(number) !== true) gaps.push(`${relative(SKILL, guide).split("\\").join("/")}:${number + 1}: ${title}`);
    }
  }
  assert(gaps.length === 0 && counted > 0, gaps.join("; "));
}

<a id="SKL-05"></a>
## SKL-05 · Every tutorial has the same steps in both languages

In [ ]:
{
  /** A tutorial's title and its steps, in order, and what is wrong with them: each step a markdown cell that starts
   * with its anchor, `<a id="Tnn.k"></a>`, then its heading, `## nn.k · title`. */
  const tutorial = (path: string) => {
    let title: string | null = null;
    const found: [string, number, string][] = [];
    const wrong: string[] = [];
    const name = path.split("/").at(-1) as string;
    for (const cell of JSON.parse(readFileSync(path, "utf8")).cells as { cell_type: string; source: string | string[] }[]) {
      const text = cell.cell_type !== "markdown" ? "" : Array.isArray(cell.source) ? cell.source.join("") : cell.source;
      title ??= /^# (.+)$/m.exec(text)?.[1] ?? null;
      const headings = [...text.matchAll(/^## \d\d\.\d+ · /gm), ...text.matchAll(/<a id="T[^"]*"><\/a>/g)];
      if (headings.length === 0) continue;
      const step = /^<a id="T(\d\d)\.(\d+)"><\/a>\n## (\d\d)\.(\d+) · (.+)$/m.exec(text);
      if (headings.length !== 2 || step === null || step.index !== 0 || step[1] !== step[3] || step[2] !== step[4]) {
        wrong.push(`${name}: a step's cell starts with its anchor, then its heading: ${JSON.stringify(text.slice(0, 60))}`);
      } else {
        found.push([step[3] as string, Number(step[4]), step[5] as string]);
      }
    }
    return { title, found, wrong };
  };
  // Each language's tutorials, by name; a step EQUIVALENCE.md lists as one language's only has no twin in the other.
  const equivalence = readFileSync(join(ROOT, "docs", "EQUIVALENCE.md"), "utf8");
  const oneLanguage = (equivalence.split("\n## Cases in one language only\n")[1] as string).split("\n## ")[0] as string;
  const ONE_LANGUAGE = new Set([...oneLanguage.matchAll(/`([A-Z]+-\d+[a-z]?|T\d\d\.\d+)`/g)].map((m) => m[1] as string));
  const TUTORIALS: Record<string, string> = Object.fromEntries(["python3", "typescript5"].map((l) => [l, join(ROOT, l, "tutorials")]));
  const names = Object.fromEntries(Object.entries(TUTORIALS).map(([language, folder]) => [language,
    readdirSync(folder).filter((n) => /^[0-9].*\.ipynb$/.test(n)).sort()])) as Record<string, string[]>;
  const problems: string[] = JSON.stringify(names.python3) === JSON.stringify(names.typescript5) ? []
    : [`not the same tutorials: ${JSON.stringify(names)}`];
  let steps = 0;
  for (const name of [...new Set([...(names.python3 as string[]), ...(names.typescript5 as string[])])].sort()) {
    const read = Object.entries(TUTORIALS).filter(([, folder]) => existsSync(join(folder, name)))
      .map(([language, folder]) => [language, tutorial(join(folder, name))] as const);
    for (const [language, { title, found, wrong }] of read) {
      problems.push(...wrong.map((w) => `${language}: ${w}`));
      if (title === null || !title.startsWith(`${Number(name.slice(0, 2))} · `)) {
        problems.push(`${language}: ${name}: its title is not numbered ${Number(name.slice(0, 2))}: ${JSON.stringify(title)}`);
      }
      problems.push(...found.filter(([nn]) => nn !== name.slice(0, 2)).map(([nn, k]) => `${language}: ${name}: step ${nn}.${k} is not numbered ${name.slice(0, 2)}`));
      steps += found.length;
    }
    const numbers = [...new Set(read.flatMap(([, { found }]) => found.map(([, k]) => k)))].sort((a, b) => a - b);
    if (numbers.some((k, i) => k !== i + 1)) problems.push(`${name}: its steps have a gap: ${numbers.join(", ")}`);
    const twins = read.map(([, { title, found }]) => JSON.stringify([title, found.filter(([nn, k]) => !ONE_LANGUAGE.has(`T${nn}.${k}`))]));
    if (twins.length === 2 && twins[0] !== twins[1]) problems.push(`${name}: not the same title and steps in both languages: ${twins.join(" against ")}`);
  }
  assert(problems.length === 0 && steps > 0, problems.join("; "));
}